In [2]:
import bs4
import urllib.request
import json


parent_link = "http://mlg.ucd.ie/modules/python/sources/cars/"
link = parent_link + "index.html"
response = urllib.request.urlopen(link)
html = response.read().decode()

parser = bs4.BeautifulSoup(html, "html.parser")

In [6]:
car_brand_list = []
for match in parser.find_all("a"):
    car_brand_list.append(match["href"]) # add all the links to a list
    
print(car_brand_list)


brand_json = {}
for brand in car_brand_list:
    brand_name = brand.split("-")[0]
    print(brand_name)
    brand_link = parent_link + brand
    brand_response = urllib.request.urlopen(brand_link)
    brand_html = brand_response.read().decode()
    first_brand_parser = bs4.BeautifulSoup(brand_html, "html.parser")

    # Find list of all the pages you have to visit
    page_list = [brand]
    for nav in first_brand_parser.find_all("nav"):
        for a in nav.find_all("a"):
            page_list.append(a["href"]) 
    page_list = list(dict.fromkeys(page_list)) # Remove any duplicates 


    cars_json = []
    for page in page_list:
        page_link = parent_link + page
        page_response = urllib.request.urlopen(page_link)
        page_html = page_response.read().decode()


        page_parser = bs4.BeautifulSoup(page_html, "html.parser")
        for match in page_parser.find_all("li"):
            car_json_dict = {}
            for name in match.find_all("h3"):
                car_name = name.get_text()
                car_json_dict["name"] = car_name
                
            for table in match.find_all("table"):
                for tr in table.find_all("tr"):
                    tr_tags = tr.find_all("td")
                    car_json_dict[tr_tags[0].get_text().strip(":")] = tr_tags[1].get_text()
            
            cars_json.append(car_json_dict) 
    brand_json[brand_name] =  cars_json

with open('data.json', 'w') as f:
    json.dump(brand_json, f)
    
    

['BMW-page01.html', 'Mercedes-Benz-page01.html', 'Toyota-page01.html', 'Volkswagen-page01.html']
BMW
Mercedes
Toyota
Volkswagen
